# Zero-Shot Prompting

Here, we'll begin the focus of week two: **prompt engineering**. Since we have our few-shot examples and setlist of skits ready, we'll start by testing zero-shot prompting exclusively in this notebook. This means:

* Giving our chosen LLM system instructions *without* any examples
* Providing a test set of user inputs, having it output a humor score, then grading it ourselves
* And recording takeaways that can be used towards few-shot prompting

For context, we'll be using Gemini 3 Flash for this step of the project.

In [9]:
# Install the official Google Generative AI SDK ('-q' silences the output, '-U' ensures the most up-to-date version)
!pip install -q -U google-generativeai

In [6]:
# Start by importing modules & stuff
import io
import time
import os
from google import genai
from google.colab import userdata
from sklearn.metrics import classification_report

# Fetch the API key here
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
client = genai.Client()

# Define the model
MODEL_ID = "gemini-3-flash-preview"

In [7]:
# Grab the CSV (Google spreadsheet of data)
import csv
from google.colab import files
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
csv_data = uploaded[file_name].decode('utf-8')

# Parse the CSV
sample_ids = list(range(1, 11)) + list(range(35, 45)) + list(range(68, 78))
sample_set = []
reader = csv.DictReader(io.StringIO(csv_data))
for row in reader:
    try:
        row_id = int(row['ID'])
        if row_id in sample_ids:
            sample_set.append({
                'id': row_id,
                'joke': row['Jokes / Bits / Puns / One-liners'],
                'actual_rating': int(row['Rating'])
            })
    except ValueError:
        continue # skip rows where ID isn't a valid integer

Saving DS 340 Final Project Data - Sheet1.csv to DS 340 Final Project Data - Sheet1.csv


In [14]:
# Create the function for zero-shot prompting
def zero_shot(dataset, model_id):
    """Passes a dataset through the LLM using a zero-shot prompt & returns lists of actual vs. predicted labels."""
    y_true = []
    y_pred = []
    print(f"Beginning the zero-shot evaluation on {len(dataset)} samples...\n")
    for item in dataset:
        prompt = f"""You are a comedy writer whose job is to evaluate how funny a given piece of text is. Classify the following examples into one of three categories: 0 (not funny), 1 (neutral), or 2 (funny). Output only the number 0, 1, or 2.
        Text: "{item['joke']}" """

        try:
            response = client.models.generate_content(
                model = model_id,
                contents = prompt,
                config = genai.types.GenerateContentConfig(
                    temperature = 0,
                )
            )
            pred_rating = int(response.text.strip()) # get the integer
            y_true.append(item['actual_rating'])
            y_pred.append(pred_rating)
            print(f"ID {item['id']} | Actual: {item['actual_rating']} | Predicted: {pred_rating}")
            time.sleep(1) # pause to respect API rate limits b/c I'm scared af
        except Exception as e:
            print(f"Error processing ID {item['id']}: {e}")

    return y_true, y_pred

In [11]:
# Run the function
true_labels, pred_labels = zero_shot(sample_set, MODEL_ID)

# Get the outputs
print("\n" + "=" * 50)
print("Zero-shot classification results")
print("=" * 50)
print(classification_report(
    true_labels,
    pred_labels,
    labels = [0, 1, 2],
    target_names = ["0 (not funny)", "1 (neutral)", "2 (funny)"],
    zero_division = 0
))

Beginning the zero-shot evaluation on 30 samples...

ID 1 | Actual: 2 | Predicted: 2
ID 2 | Actual: 2 | Predicted: 2
ID 3 | Actual: 2 | Predicted: 2
ID 4 | Actual: 2 | Predicted: 2
ID 5 | Actual: 2 | Predicted: 2
ID 6 | Actual: 2 | Predicted: 2
ID 7 | Actual: 2 | Predicted: 2
ID 8 | Actual: 2 | Predicted: 2
Error processing ID 9: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
ID 10 | Actual: 2 | Predicted: 2
ID 35 | Actual: 1 | Predicted: 2
ID 36 | Actual: 1 | Predicted: 2
ID 37 | Actual: 1 | Predicted: 2
ID 38 | Actual: 1 | Predicted: 2
ID 39 | Actual: 1 | Predicted: 2
ID 40 | Actual: 1 | Predicted: 2
ID 41 | Actual: 1 | Predicted: 2
ID 42 | Actual: 1 | Predicted: 2
ID 43 | Actual: 1 | Predicted: 2
Error processing ID 44: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand